# 00 · Hello, GPU

**Teach a Small AI to Think** · unicodesnwrites.tech

In this notebook you will switch on a free GPU, install the libraries the course uses, load a small open model (Qwen2.5-0.5B-Instruct) and ask it a few grade-school maths questions. You will see that it often gets them wrong. Fixing that is what the rest of the course is about.

**How to open it in Colab**

1. Go to [colab.research.google.com](https://colab.research.google.com) and sign in with your Google account.
2. Choose **File → Upload notebook** and pick this `.ipynb` file.
3. Choose **Runtime → Change runtime type**, select **T4 GPU**, and click **Save**.
4. Run each cell in order with **Shift + Enter**.

**Expected runtime on a free T4:** about 5–8 minutes, most of it installing libraries and downloading the model (about 1 GB).

## 1. Is there a GPU?

`nvidia-smi` is a small tool that lists the graphics cards the computer can see. You should see a **Tesla T4** with about 15 GB of memory. If you see an error instead, go back to step 3 above and pick the T4 runtime.

In [ ]:
!nvidia-smi

## 2. Install the course libraries

We pin exact versions so the notebook behaves the same for everyone, even after the libraries change. These are the versions the course was written against:

| library | what it does |
|---|---|
| `transformers` | loads and runs models |
| `accelerate` | moves work onto the GPU efficiently |
| `datasets` | downloads datasets such as GSM8K |
| `trl` | the reinforcement learning trainers (we use it from Module 5) |

If Colab asks you to **restart the session** after this cell, click the button, then carry on from the next cell.

In [ ]:
%pip install -q transformers==5.20.2 accelerate==1.15.0 datasets==5.1.0 trl==1.16.0

In [ ]:
import torch, transformers, datasets, trl

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| datasets", datasets.__version__, "| trl", trl.__version__)

## 3. Load the model

Qwen2.5-0.5B-Instruct has about half a billion parameters. That is tiny by today's standards, which is exactly why it fits on a free GPU and why it still has a lot to learn. `float16` stores each number in half the usual space so the model loads faster and uses less memory.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side="left")
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16).to("cuda")
model.eval()
print(f"Loaded {MODEL_ID} with {model.num_parameters() / 1e6:.0f}M parameters")

## 4. Meet GSM8K

GSM8K is a set of about 8,500 grade-school maths word problems. Each one comes with a worked solution, and the final number always comes after `####`. That makes it easy for a computer to check an answer, which is the key idea of Module 2.

In [ ]:
from datasets import load_dataset

gsm8k = load_dataset("openai/gsm8k", "main", split="test")
example = gsm8k[0]
print(example["question"])
print("\nGold solution:\n" + example["answer"])

## 5. Ask the model one question

We wrap the question in the model's chat format, ask it to show its working, and let it write up to 400 new tokens (a token is roughly three quarters of a word). `do_sample=False` means it always picks its single most likely next token, so you get the same answer every time.

In [ ]:
def ask(questions, max_new_tokens=400):
    prompts = [
        tokenizer.apply_chat_template(
            [{"role": "user", "content": q + "\nShow your working, then give the final number on its own line."}],
            tokenize=False, add_generation_prompt=True)
        for q in questions
    ]
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = out[:, inputs["input_ids"].shape[1]:]
    return tokenizer.batch_decode(new_tokens, skip_special_tokens=True)

reply = ask([example["question"]])[0]
print(reply)

## 6. Was it right?

We grade the reply the simple way: take the **last number** the model wrote and compare it with the number after `####`. This is rough, but it is the same kind of automatic check you will build properly in Module 2.

In [ ]:
import re

def last_number(text):
    numbers = re.findall(r"-?\d+(?:\.\d+)?", text.replace(",", ""))
    return float(numbers[-1]) if numbers else None

def gold_number(solution):
    return float(solution.split("####")[-1].strip().replace(",", ""))

print("model said:", last_number(reply), "| gold:", gold_number(example["answer"]))

## 7. Twenty questions

One question can be luck either way. Let us ask twenty at once (batched, so it takes about a minute) and count. Independent measurements put this model somewhere around a third to a half right on GSM8K, depending on the prompt and how answers are graded, so expect plenty of red crosses. Your exact score may differ slightly from a friend's because GPUs round numbers in slightly different orders.

In [ ]:
batch = gsm8k.select(range(20))
replies = ask(batch["question"])

correct = 0
for i, (reply, solution) in enumerate(zip(replies, batch["answer"])):
    ok = last_number(reply) == gold_number(solution)
    correct += ok
    print(f"{i:2d}  {'✓' if ok else '✗'}  model: {last_number(reply)}  gold: {gold_number(solution)}")
print(f"\n{correct}/20 correct")

## 8. Look at a wrong one

Pick a ✗ from the list above and read the full reply. Common problems: it skips a step, copies a number wrongly, stops half-way, or never states a final number at all.

In [ ]:
wrong = [i for i, (r, s) in enumerate(zip(replies, batch["answer"])) if last_number(r) != gold_number(s)]
if wrong:
    i = wrong[0]
    print(batch["question"][i], "\n\n--- model reply ---\n", replies[i], "\n\n--- gold ---\n", batch["answer"][i])
else:
    print("All 20 right! Lucky batch. Try gsm8k.select(range(20, 40)) instead.")

## You're set up

If you saw a T4, loaded the model and got a score out of 20, your setup works. Write your score down: at the end of Module 5 you will run this same check on the model you trained and compare.

Next lesson: **Answer now, or think first?**